In [17]:
import json
import re
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from typing import List, Dict, Any

class RussianSentenceSplitter:
    """
    Класс для разделения сложных предложений русского языка на простые
    с помощью LLM и строгого JSON-вывода.
    """
    def __init__(self, model_name: str = "Qwen/Qwen2.5-7B-Instruct", device: str = "auto"):
        """
        :param model_name: HuggingFace ID instruct-модели с поддержкой русского языка
        :param device: 'auto', 'cuda', 'cpu' и т.д.
        """
        print(f"🔄 Загрузка модели {model_name}...")
        self.pipe = pipeline(
            "text-generation",
            model=model_name,
            tokenizer=model_name,
            device_map=device,
            torch_dtype="auto",
            trust_remote_code=True
        )
        if not self.pipe.tokenizer.chat_template:
            raise ValueError("Модель не поддерживает chat_template. Используйте instruct-версию.")

    def _build_prompt(self, sentence: str) -> List[Dict[str, str]]:
        """Формирует системный и пользовательский промпт"""
        return [
            {
                "role": "system",
                "content": (
                    "Ты — лингвистический ассистент. Твоя задача: разделить сложное предложение "
                    "русского языка на простые грамматически законченные части. "
                    "Сохраняй исходную лексику и порядок слов. Если в предложении есть противопоставление "
                    "или причинно-следственная связь, разделяй его на две части, сохраняя смысл. "
                    "Верни результат СТРОГО в формате JSON без markdown, пояснений или лишнего текста:\n"
                    '{"simple_parts": ["часть 1", "часть 2"]}'
                )
            },
            {"role": "user", "content": f"Раздели на простые: \"{sentence}\""}
        ]

    def _extract_json(self, raw_text: str) -> List[str]:
        """Безопасно извлекает JSON из ответа модели"""
        # Убираем markdown-обёртки если есть
        cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw_text.strip(), flags=re.MULTILINE)
        # Ищем первый валидный JSON-объект
        match = re.search(r'\{[\s\S]*\}', cleaned)
        if not match:
            return []
        try:
            data = json.loads(match.group())
            return data.get("simple_parts", [])
        except json.JSONDecodeError:
            return []

    def split_sentence(self, sentence: str) -> Dict[str, Any]:
        """Разделяет одно предложение"""
        messages = self._build_prompt(sentence)
        input_text = self.pipe.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        outputs = self.pipe(
            input_text,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.1,
            top_p=1.0,
            return_full_text=False,
            pad_token_id=self.pipe.tokenizer.eos_token_id
        )
        response = outputs[0]["generated_text"].strip()
        parts = self._extract_json(response)

        return {"original": sentence, "simple_parts": parts}

    def split_dataset(self, sentences: List[str], output_path: str = None) -> pd.DataFrame:
        """Обрабатывает датасет, сохраняет прогресс и результат"""
        results = []
        for sent in tqdm(sentences):
            try:
                result = self.split_sentence(sent)
                result["status"] = "ok"
            except Exception as e:
                result = {"original": sent, "simple_parts": [], "error": str(e), "status": "error"}
            results.append(result)

        df = pd.DataFrame(results)
        if output_path:
            df.to_json(output_path, orient="records", force_ascii=False, indent=2)
            print(f"💾 Результат сохранён: {output_path}")
        return df

In [ ]:
# 1. Тестовый датасет (в реальном проекте загрузите из CSV/JSONL/Parquet)
test_sentences = [
    "Я не хочу жить, но мне жалко родителей.",
    "Всё бессмысленно, однако я пока не готов уйти.",
    "Мне больно, потому что меня никто не понимает.",
    "Я устал бороться, но обещаю попробовать ещё раз.",
    "Если бы меня не было, всем стало бы легче, поэтому я молчу."
]

# 2. Инициализация сплиттера
splitter = RussianSentenceSplitter(
    model_name="Qwen/Qwen2.5-7B-Instruct",  # или "ai-forever/ruGPT-3.5-13B", "meta-llama/Llama-3.1-8B-Instruct"
    device="cuda"  # используйте "cpu" если нет GPU
)

# 3. Запуск
df_results = splitter.split_dataset(test_sentences, output_path="split_results.json")

# 4. Вывод
for _, row in df_results.iterrows():
    print(f"📝 Оригинал: {row['original']}")
    print(f"🔪 Части: {row['simple_parts']}")
    print("-" * 50)

🔄 Загрузка модели Qwen/Qwen2.5-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


🔍 Разделение предложений:   0%|          | 0/5 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'top_p', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

🔍 Разделение предложений:  20%|██        | 1/5 [00:03<00:13,  3.50s/it]Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (

💾 Результат сохранён: split_results.json
📝 Оригинал: Я не хочу жить, но мне жалко родителей.
🔪 Части: ['Я не хочу жить.', 'мне жалко родителей.']
--------------------------------------------------
📝 Оригинал: Всё бессмысленно, однако я пока не готов уйти.
🔪 Части: ['Всё бессмысленно.', 'Однако я пока не готов уйти.']
--------------------------------------------------
📝 Оригинал: Мне больно, потому что меня никто не понимает.
🔪 Части: ['Мне больно.', 'Потому что меня никто не понимает.']
--------------------------------------------------
📝 Оригинал: Я устал бороться, но обещаю попробовать ещё раз.
🔪 Части: ['Я устал бороться.', 'Но обещаю попробовать ещё раз.']
--------------------------------------------------
📝 Оригинал: Если бы меня не было, всем стало бы легче, поэтому я молчу.
🔪 Части: ['Если бы меня не было, всем стало бы легче', 'я молчу']
--------------------------------------------------


In [12]:
data = pd.read_csv("data.csv")
data = data[~data['simple_sentences'].isna()][['text', 'simple_sentences']]
# data['simple_sentences'] = data['simple_sentences'].apply(lambda x: x['text'])
data

,text,simple_sentences
0,"Грустно, ведь мне не с кем даже посмеяться над...","{""text"":[""Грустно."",""Ведь мне не с кем даже по..."
1,Сегодня вечером я плачу от того что живу в Рос...,"{""text"":[""Сегодня вечером я плачу от того."",""(..."
2,"И прошло пару лет, и я в ужасной депрессии и н...","{""text"":[""Прошло пару лет."",""Я в ужасной депре..."
3,"Одна, со мной обожающая меня маленькая очарова...","{""text"":[""Одна."",""Со мной обожающая меня мален..."
5,"И очень сильно устаю: у меня нет сил, эмоций, ...","{""text"":[""И очень сильно устаю."",""У меня нет с..."
...,...,...
115,"она меня удалила везде, где только можно и не ...","{""text"":[""Она меня удалила везде, где только м..."
116,"меня все бесит, с одноклассниками всгда пробле...","{""text"":[""Меня всё бесит."",""С одноклассниками ..."
117,Может это расплата за какие -то мои грехи о ко...,"{""text"":[""Может, это расплата за какие-то мои ..."
118,"Я уже не знаю как, но пока живу, через силу, п...","{""text"":[""Я уже не знаю как, но пока живу."",""Ж..."


In [14]:
texts_100 = data['text'].to_list()
texts_100[:2]

['Грустно, ведь мне не с кем даже посмеяться над глупой шуткой.',
 'Сегодня вечером я плачу от того что живу в России.']

In [16]:
df_results = splitter.split_dataset(texts_100, output_path="split_results_100.json")

# 4. Вывод
for _, row in df_results.iterrows():
    print(f"📝 Оригинал: {row['original']}")
    print(f"🔪 Части: {row['simple_parts']}")
    print("-" * 50)


🔍 Разделение предложений:   0%|          | 0/96 [00:00<?, ?it/s]Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

🔍 Разделение предложений:   1%|          | 1/96 [00:03<05:20,  3.38s/it]Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

🔍 Разделение предложений:   2%|▏         | 2/96 [00:05<04:34,  2.92s/it]Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

🔍 Разделение предложений:   3%|▎         

💾 Результат сохранён: split_results_100.json
📝 Оригинал: Грустно, ведь мне не с кем даже посмеяться над глупой шуткой.
🔪 Части: ['Грустно, ведь мне не с кем даже посмеяться.', 'над глупой шуткой.']
--------------------------------------------------
📝 Оригинал: Сегодня вечером я плачу от того что живу в России.
🔪 Части: ['Сегодня вечером я плачу.', 'от того что живу в России.']
--------------------------------------------------
📝 Оригинал: И прошло пару лет, и я в ужасной депрессии и не хочу жить.
🔪 Части: ['И прошло пару лет.', 'я в ужасной депрессии и не хочу жить.']
--------------------------------------------------
📝 Оригинал: Одна, со мной обожающая меня маленькая очаровательная собачка, обязательства по кредитам.
🔪 Части: ['Одна, со мной обожающая меня маленькая очаровательная собачка.', 'Обязательства по кредитам.']
--------------------------------------------------
📝 Оригинал: И очень сильно устаю: у меня нет сил, эмоций, энергии и желания что-либо делать.
🔪 Части: ['И очень сил

In [24]:
golds_100 = []
for i, r in data.iterrows():
    try:
        golds_100.append(eval(r['simple_sentences'])['text'])
    except:
        golds_100.append([r['simple_sentences']])


In [25]:
# golds_100 = [r['simple_sentences']['text'] for i, r in data.iterrows()]
preds = [row['simple_parts'] for i, row in df_results.iterrows()]

In [26]:
for gold, pred in zip(golds_100, preds):
    print(f"📝 Оригинал: {gold}")
    print(f"🔪 Части: {pred}")
    print("-" * 50)

📝 Оригинал: ['Грустно.', 'Ведь мне не с кем даже посмеяться над глупой шуткой.']
🔪 Части: ['Грустно, ведь мне не с кем даже посмеяться.', 'над глупой шуткой.']
--------------------------------------------------
📝 Оригинал: ['Сегодня вечером я плачу от того.', '(что) живу в России.']
🔪 Части: ['Сегодня вечером я плачу.', 'от того что живу в России.']
--------------------------------------------------
📝 Оригинал: ['Прошло пару лет.', 'Я в ужасной депрессии и не хочу жить.']
🔪 Части: ['И прошло пару лет.', 'я в ужасной депрессии и не хочу жить.']
--------------------------------------------------
📝 Оригинал: ['Одна.', 'Со мной обожающая меня маленькая очаровательная собачка.', '(Со мной) обязательства по кредитам.']
🔪 Части: ['Одна, со мной обожающая меня маленькая очаровательная собачка.', 'Обязательства по кредитам.']
--------------------------------------------------
📝 Оригинал: ['И очень сильно устаю.', 'У меня нет сил, эмоций, энергии и желания что-либо делать.']
🔪 Части: ['И очень с